In [1]:
import os
from os.path import join
import sys
from pathlib import Path
import time

import numpy as np
import viser
import smplx

import torch
import trimesh
from mhr.mhr import MHR

from conversion import Conversion

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
server = viser.ViserServer()

server.scene.add_grid(
    "/grid",
    width=10.0,
    height=10.0,
    position=np.array([0.0, 0.0, 0.0]),
)

╭────── viser (listening *:8080) ───────╮
│             ╷                         │
│   HTTP      │ http://localhost:8080   │
│   Websocket │ ws://localhost:8080     │
│             ╵                         │
╰───────────────────────────────────────╯

GridHandle(width=10.0, height=10.0, plane='xy', cell_color=(200, 200, 200), cell_thickness=1.0, cell_size=0.5, section_color=(140, 140, 140), section_thickness=1.0, section_size=1.0, infinite_grid=False, fade_distance=100.0, fade_strength=1.0, fade_from='camera', shadow_opacity=0.125)

(viser) Connection opened (0, 1 total), 7 persistent messages

In [4]:
mhr_model = MHR.from_files(folder=Path("../../assets"), device=device, lod=1)

smplx_model = smplx.SMPLX(
    model_path="../../../better_human/models/smplx/SMPLX_NEUTRAL.npz", 
    gender="neutral", num_betas=16)

converter = Conversion(
    mhr_model=mhr_model,
    smpl_model=smplx_model,
    method="pytorch",  # or "pymomentum"
    batch_size=128,
)

In [3]:
amass_path = "/media/rikhat/Hard/datasets/AMASS/SMPL-X_N/"
# data_folder = "DanceDB/20120731_StefanosTheodorou"
# filename = "Stefanos_2os_antrikos_karsilamas_C3D_poses.npz"

data_folder = "ACCAD/Female1General_c3d"
filename = "A10_-_lie_to_crouch_stageii.npz"

data = np.load(join(amass_path, data_folder, filename), allow_pickle=True)

for k in data:
    print(k, data[k].shape)

print(data['mocap_frame_rate'])
print(data['gender'])

gender ()
surface_model_type ()
mocap_frame_rate ()
mocap_time_length ()
markers_latent (41, 3)
latent_labels (41,)
markers_latent_vids ()
trans (524, 3)
poses (524, 165)
betas (16,)
num_betas ()
root_orient (524, 3)
pose_body (524, 63)
pose_hand (524, 90)
pose_jaw (524, 3)
pose_eye (524, 6)
120.0
neutral


In [6]:
fps = 60
start = 0
end = 600

data_framerate = data['mocap_frame_rate']
batch_size = 1
seq_len = int(data['poses'].shape[0] / data_framerate * fps)
indices = np.linspace(0, data['poses'].shape[0]-1, seq_len).astype(np.int32)
# indices = indices[start:end]
seq_len = indices.shape[0]
print("seq_len", seq_len)

smplx_params = {
    "betas": torch.tensor(data['betas'], device=device, dtype=torch.float32).unsqueeze(0).repeat(seq_len, 1),
    "global_orient": torch.tensor(data['root_orient'][indices], device=device, dtype=torch.float32),
    "body_pose": torch.tensor(data['pose_body'][indices], device=device, dtype=torch.float32),
    "transl": torch.tensor(data['trans'][indices], device=device, dtype=torch.float32),
}

# results = converter.convert_smpl2mhr(
#     smpl_parameters=smplx_params,
#     single_identity=True,
#     exclude_expression=True,
#     return_mhr_meshes=False,
#     return_mhr_parameters=True,
#     is_tracking=True,
# )

seq_len 262


In [6]:
vertices, skeleton_state = mhr_model(
    results.result_parameters["identity_coeffs"],
    results.result_parameters["lbs_model_params"], 
    torch.zeros(seq_len, 72, device=device) 
)

for i in range(seq_len):
    mhr_mesh = server.scene.add_mesh_simple(
        "/mhr",
        vertices=vertices[i].detach().cpu().numpy() / 100.0,  # Convert cm to meters
        faces=mhr_model.character.mesh.faces,
        opacity=1.0
    )

    time.sleep(0.025)


In [8]:
smplx_output = smplx_model(
    global_orient=smplx_params["global_orient"],
    body_pose=smplx_params["body_pose"],
    betas=smplx_params["betas"],
    transl=smplx_params["transl"],
    expression=torch.zeros((seq_len, 10), dtype=torch.float32, device=device),
    left_hand_pose=torch.zeros((seq_len, 6), dtype=torch.float32, device=device),
    right_hand_pose=torch.zeros((seq_len, 6), dtype=torch.float32, device=device),
    jaw_pose=torch.zeros((seq_len, 3), dtype=torch.float32, device=device),
    leye_pose=torch.zeros((seq_len, 3), dtype=torch.float32, device=device),
    reye_pose=torch.zeros((seq_len, 3), dtype=torch.float32, device=device),
)

for i in range(seq_len):
    smplx_mesh = server.scene.add_mesh_simple(
        "/smplx",
        vertices=smplx_output.vertices[i].detach().cpu().numpy(),
        faces=smplx_model.faces,
        opacity=1.0,
        color=[20, 200, 10]
    )

    time.sleep(0.025)